<a href="https://colab.research.google.com/github/azraisik/yapay-sinir-aglari/blob/kerem%2Ftasar%C4%B1m-mimari/TASARIM_VE_MIMARI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Adim: Kutuphaneleri projemize dahil ediyoruz
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)

# 2. Adim: Temizlenmis veri setimizi iceri aktariyoruz
df = pd.read_csv("SLA_Temiz_Veri.csv")

# Verimizin ilk 5 satirina goz atip dogru yukledik mi diye kontrol edelim
print("--- Veri Setinin Ilk 5 Satiri ---")
print(df.head())


In [ ]:
# 'Case ID' sutunu tahminde bir ise yaramayacagi icin onu tablodan tamamen siliyoruz
df = df.drop(columns=['Case ID'], errors='ignore')
print("\n--- Case ID Silindikten Sonra Kalan Sutunlar ---")
print(df.columns)


In [ ]:
# Dilara bolumundeki veri hazirlik yaklasimiyla uyumlu olacak sekilde kolonlari ayiriyoruz.
categorical_features = ['Variant', 'Priority', 'Issue Type', 'Report Channel']
numeric_features = ['Step_count', 'Reassignment_count', 'Escalation_count',
                    'Has_Bounce', 'Workload_Index', 'Open_Hour', 'Is_Weekend']
model_features = categorical_features + numeric_features

# Yapay zekaya girecek ozellikleri (X) ve tahmin edilmek istenen hedefi (y) ayiralim.
# Hedefimiz: SLA_Violation (ihlal var=1, yok=0)
X = df[model_features]
y = df['SLA_Violation'].astype(int)

print(f"\nHam giris verisinin (X) boyutu (Satir, Sutun): {X.shape}")
print("Hedef degisken sinif dagilimi:")
print(y.value_counts())


In [ ]:
# 1. Veriyi %80 egitim, %20 test olarak boluyoruz.
# SLA ihlali sinifi az oldugu icin stratify=y ile sinif oranini iki sette de koruyoruz.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 2. Veri pipeline'i: sayisal kolonlar olceklenir, kategorik kolonlar One-Hot Encoding ile donusturulur.
# drop='first', onceki get_dummies(drop_first=True) mantigiyla tutarlidir.
try:
    one_hot_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)

column_transformer_args = {
    'transformers': [
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', one_hot_encoder, categorical_features)
    ],
    'remainder': 'drop'
}

try:
    preprocessor = ColumnTransformer(**column_transformer_args, verbose_feature_names_out=False)
except TypeError:
    preprocessor = ColumnTransformer(**column_transformer_args)

data_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# Pipeline sadece egitim verisine fit edilir; test seti ayni kurallarla donusturulur.
X_train_scaled = data_pipeline.fit_transform(X_train).astype('float32')
X_test_scaled = data_pipeline.transform(X_test).astype('float32')

# Dengesiz sinif dagilimini egitimde telafi etmek icin sinif agirliklari hesaplanir.
class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {
    int(class_label): float(weight)
    for class_label, weight in zip(np.unique(y_train), class_weight_values)
}

print("Egitim setindeki ornek sayisi:", X_train_scaled.shape[0])
print("Test setindeki ornek sayisi:", X_test_scaled.shape[0])
print("Giris yapacak toplam ozellik (sutun) sayisi:", X_train_scaled.shape[1])
print("\nEgitim seti sinif dagilimi:")
print(y_train.value_counts())
print("\nTest seti sinif dagilimi:")
print(y_test.value_counts())
print("\nModel egitiminde kullanilacak sinif agirliklari:")
print(class_weights)


In [ ]:
# Sirali bir model zinciri olusturuyoruz
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),

    # 1. gizli katman: 64 noronlu, aktivasyon fonksiyonu ReLU.
    Dense(64, activation='relu'),
    # Dropout egitim sirasinda noronlarin %20'sini rastgele kapatarak ezberlemeyi azaltir.
    Dropout(0.2),

    # 2. gizli katman: 32 noronlu, yine ReLU aktivasyonlu.
    Dense(32, activation='relu'),
    Dropout(0.2),

    # Cikis katmani: ikili siniflandirma icin sigmoid olasilik uretir.
    Dense(1, activation='sigmoid')
])

# Modeli derlerken accuracy yaninda dengesiz veri icin daha anlamli metrikleri de takip ediyoruz.
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

print("\n--- YAPAY SINIR AGI MIMARI OZETI ---")
model.summary()


In [ ]:
print("\n--- MODEL EGITIMI BASLIYOR ---")

# EarlyStopping, validation loss iyilesmediginde egitimi durdurur ve en iyi agirliklari geri yukler.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Modeli egitiyoruz ve egitim gecmisini 'history' degiskenine kaydediyoruz.
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

print(f"\nEgitim tamamlandi. Calisilan epoch sayisi: {len(history.history['loss'])}")
print(f"En dusuk validation loss: {min(history.history['val_loss']):.4f}")
print(f"En yuksek validation recall: {max(history.history['val_recall']):.4f}")
print(f"En yuksek validation AUC: {max(history.history['val_auc']):.4f}")

# Accuracy tek basina yeterli olmadigi icin test setinde ek siniflandirma metrikleri hesapliyoruz.
y_pred_prob = model.predict(X_test_scaled).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\n--- TEST SETI METRIK OZETI (Threshold = 0.50) ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"AUC      : {roc_auc_score(y_test, y_pred_prob):.4f}")

# Precision dusuk kaldiginda karar esigi artirilabilir; bu false positive sayisini azaltir.
threshold_results = []
for threshold in np.arange(0.30, 0.91, 0.05):
    threshold_pred = (y_pred_prob >= threshold).astype(int)
    threshold_results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_test, threshold_pred),
        'Precision': precision_score(y_test, threshold_pred, zero_division=0),
        'Recall': recall_score(y_test, threshold_pred, zero_division=0),
        'F1-Score': f1_score(y_test, threshold_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
best_threshold_row = threshold_df.sort_values('F1-Score', ascending=False).iloc[0]

print("\n--- THRESHOLD DENEMELERI ---")
display(threshold_df.style.format({
    'Threshold': '{:.2f}',
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}))

print("\n--- F1-SCORE'A GORE EN DENGELI THRESHOLD ---")
print(f"Threshold: {best_threshold_row['Threshold']:.2f}")
print(f"Precision: {best_threshold_row['Precision']:.4f}")
print(f"Recall   : {best_threshold_row['Recall']:.4f}")
print(f"F1-Score : {best_threshold_row['F1-Score']:.4f}")


In [ ]:
# YSA modelini Streamlit projesinde kullanmak icin gerekli dosyalari disari aktarir.
# Bu hucreyi model egitimi tamamlandiktan sonra calistirin.
import json
import joblib
import zipfile
from pathlib import Path

EXPORT_DIR = Path('.')
MODEL_FILE = 'ysa_sla_risk_model.keras'
NUMPY_MODEL_FILE = 'ysa_sla_risk_weights.npz'
PREPROCESSOR_FILE = 'ysa_preprocessor.pkl'
FEATURE_LIST_FILE = 'ysa_feature_list.json'
CONFIG_FILE = 'ysa_model_config.json'
ZIP_FILE = 'ysa_artifacts.zip'

# 1. Egitilmis Keras modelini kaydet.
model.save(EXPORT_DIR / MODEL_FILE)

# TensorFlow kurulu olmayan Streamlit ortamlarinda kullanmak icin Dense katman agirliklarini NumPy formatinda kaydet.
dense_weights = {}
dense_layer_index = 0
for layer in model.layers:
    weights = layer.get_weights()
    if len(weights) == 2:
        dense_weights[f'W{dense_layer_index}'] = weights[0]
        dense_weights[f'b{dense_layer_index}'] = weights[1]
        dense_layer_index += 1
np.savez(EXPORT_DIR / NUMPY_MODEL_FILE, **dense_weights)

# 2. Egitimde fit edilen preprocessing pipeline'ini kaydet.
# Manuel tahminde kategorik encoding ve sayisal scaling ayni sirayla uygulanir.
joblib.dump(data_pipeline, EXPORT_DIR / PREPROCESSOR_FILE)

# 3. Donusturulmus kolon siralarini kaydet.
try:
    processed_features = data_pipeline.named_steps['preprocessor'].get_feature_names_out().tolist()
except Exception:
    processed_features = []

with open(EXPORT_DIR / FEATURE_LIST_FILE, 'w', encoding='utf-8') as file:
    json.dump(processed_features, file, ensure_ascii=False, indent=2)

# 4. app.py tarafinin okuyacagi YSA config dosyasini olustur.
ysa_config = {
    'model_file': MODEL_FILE,
    'numpy_model_file': NUMPY_MODEL_FILE,
    'case_file': 'SLA_Arayuz_Dosyalari.csv',
    'training_file': 'SLA_Temiz_Veri.csv',
    'preprocessor_file': PREPROCESSOR_FILE,
    'feature_list_file': FEATURE_LIST_FILE,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'model_features': model_features,
    'target': 'SLA_Violation'
}

with open(EXPORT_DIR / CONFIG_FILE, 'w', encoding='utf-8') as file:
    json.dump(ysa_config, file, ensure_ascii=False, indent=2)

# 5. Tum dosyalari tek zip haline getir.
with zipfile.ZipFile(EXPORT_DIR / ZIP_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
    for file_name in [MODEL_FILE, NUMPY_MODEL_FILE, PREPROCESSOR_FILE, FEATURE_LIST_FILE, CONFIG_FILE]:
        zip_file.write(EXPORT_DIR / file_name, arcname=file_name)

print('YSA dosyalari olusturuldu:')
for file_name in [MODEL_FILE, NUMPY_MODEL_FILE, PREPROCESSOR_FILE, FEATURE_LIST_FILE, CONFIG_FILE, ZIP_FILE]:
    print('-', file_name)


try:
    from google.colab import files
    files.download(str(EXPORT_DIR / ZIP_FILE))
except Exception:
    print(f'Zip hazir: {ZIP_FILE}. Bu dosyayi indirip Streamlit proje klasorune acin.')
